# PrepSense — Data Retrieval Exploration
**Week 1 · Validating data sources before structuring the project**

Data sources used:
- **OpenWeatherMap One Call API 3.0** — current conditions + national alerts (AEMET for Spain)
- **OpenWeatherMap Forecast API 5 days/3h** — extended forecast (free tier)
- **SQLite** — local database schema validation

> One Call 3.0 requires the 'One Call by Call' subscription (free up to 1,000 calls/day).  
> Set your daily limit at: https://home.openweathermap.org/subscriptions

## 0. Setup

In [237]:
%pip install requests python-dotenv pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [238]:
import requests
import json
import sqlite3
import pandas as pd
from datetime import datetime
from pprint import pprint

print('Imports OK')

Imports OK


## 1. Configuration

Register your free API key at: https://openweathermap.org/api  
Activate One Call 3.0 at: https://home.openweathermap.org/subscriptions  

Load from a `.env` file

In [239]:
# Create a .env file with: OWM_API_KEY=your_key
from dotenv import load_dotenv
import os
load_dotenv()
OWM_API_KEY = os.getenv('OWM_API_KEY')

# Location
CITY  = 'Zaragoza'
LAT   = 41.64937507947522 
LON   = -0.8937800462407269
UNITS = 'metric'

print(f'Config loaded | City: {CITY} | lat={LAT}, lon={LON}')

Config loaded | City: Zaragoza | lat=41.64937507947522, lon=-0.8937800462407269


---
## 2. One Call API 3.0 — Current Conditions + National Alerts

Single endpoint returning current weather **and** active national alerts from AEMET (Spain).  
Docs: https://openweathermap.org/api/one-call-3

In [240]:
def get_onecall(api_key, lat, lon, units='metric'):
    """
    One Call API 3.0 — returns current conditions and national alerts.
    minutely/hourly/daily excluded to minimise payload.
    """
    url = 'https://api.openweathermap.org/data/3.0/onecall'
    params = {
        'lat':     lat,
        'lon':     lon,
        'appid':   api_key,
        'units':   units,
        'exclude': 'minutely,hourly,daily',
    }
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


raw_onecall = get_onecall(OWM_API_KEY, LAT, LON)

print('One Call API 3.0 response received')
print(f'Keys in response: {list(raw_onecall.keys())}')
print(f'Active alerts:    {len(raw_onecall.get("alerts", []))}')

One Call API 3.0 response received
Keys in response: ['lat', 'lon', 'timezone', 'timezone_offset', 'current', 'alerts']
Active alerts:    1


In [241]:
def parse_current(raw):
    """Extract relevant fields from One Call current block."""
    c = raw.get('current', {})
    return {
        'timestamp':           datetime.utcfromtimestamp(c['dt']).isoformat(),
        'city':                raw.get('timezone', '').split('/')[-1],
        'temp_c':              c.get('temp'),
        'feels_like_c':        c.get('feels_like'),
        'humidity_pct':        c.get('humidity'),
        'pressure_hpa':        c.get('pressure'),
        'visibility_m':        c.get('visibility'),
        'wind_speed_ms':       c.get('wind_speed'),
        'wind_gust_ms':        c.get('wind_gust'),
        'rain_1h_mm':          c.get('rain', {}).get('1h', 0),
        'snow_1h_mm':          c.get('snow', {}).get('1h', 0),
        'weather_id':          c.get('weather', [{}])[0].get('id'),
        'weather_main':        c.get('weather', [{}])[0].get('main'),
        'weather_description': c.get('weather', [{}])[0].get('description'),
        'clouds_pct':          c.get('clouds'),
        'uvi':                 c.get('uvi'),
    }


weather = parse_current(raw_onecall)

print('Parsed current conditions:\n')
for k, v in weather.items():
    print(f'  {k:<28} {v}')

Parsed current conditions:

  timestamp                    2026-04-10T14:09:16
  city                         Madrid
  temp_c                       27.34
  feels_like_c                 26.36
  humidity_pct                 22
  pressure_hpa                 1015
  visibility_m                 10000
  wind_speed_ms                1.03
  wind_gust_ms                 None
  rain_1h_mm                   0
  snow_1h_mm                   0
  weather_id                   800
  weather_main                 Clear
  weather_description          clear sky
  clouds_pct                   0
  uvi                          4.64


/tmp/ipykernel_68017/523335232.py:5: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'timestamp':           datetime.utcfromtimestamp(c['dt']).isoformat(),


In [242]:
# View as DataFrame
pd.DataFrame([weather]).T.rename(columns={0: 'value'})

,value
timestamp,2026-04-10T14:09:16
city,Madrid
temp_c,27.34
feels_like_c,26.36
humidity_pct,22
pressure_hpa,1015
visibility_m,10000
wind_speed_ms,1.03
wind_gust_ms,None
rain_1h_mm,0


---
## 3. One Call API 3.0 — National Alerts

Alerts are issued by **AEMET** for Spain.  

Note: One Call 3.0 does not expose severity as a label (Yellow/Orange/Red) directly —  
we infer it from the `tags` field and feed it into the risk engine.

In [243]:
def infer_severity_from_tags(tags):
    """
    Infer severity label from One Call alert tags.
    Returns: Minor / Moderate / Severe / Extreme / Unknown
    """
    tags_lower = [t.lower() for t in (tags or [])]
    if any(t in tags_lower for t in ['extreme', 'tornado', 'hurricane', 'tsunami']):
        return 'Extreme'
    elif any(t in tags_lower for t in ['thunderstorm', 'snow', 'flood', 'wind', 'rain', 'fog']):
        return 'Moderate'
    elif tags:
        return 'Minor'
    return 'Unknown'


def parse_alerts(raw):
    """Extract and normalise alerts from One Call 3.0 response."""
    parsed = []
    for a in raw.get('alerts', []):
        tags     = a.get('tags', [])
        severity = infer_severity_from_tags(tags)
        parsed.append({
            'title':       a.get('event', ''),
            'event':       a.get('event', ''),
            'sender_name': a.get('sender_name', ''),
            'severity':    severity,
            'urgency':     'Expected',
            'certainty':   'Likely',
            'onset':       datetime.utcfromtimestamp(a['start']).isoformat() if 'start' in a else '',
            'expires':     datetime.utcfromtimestamp(a['end']).isoformat()   if 'end'   in a else '',
            'description': a.get('description', ''),
            'tags':        tags,
        })
    return parsed


alerts = parse_alerts(raw_onecall)

print(f'Active alerts: {len(alerts)}')
if alerts:
    for a in alerts:
        print(f"\n  [{a['sender_name']}] {a['event']}")
        print(f"  Severity : {a['severity']}")
        print(f"  Tags     : {a['tags']}")
        print(f"  Onset    : {a['onset']}")
        print(f"  Expires  : {a['expires']}")
        print(f"  Desc     : {a['description'][:160]}")
else:
    print('  No active alerts — normal under calm conditions.')

Active alerts: 1

  [AEMET. Agencia Estatal de Meteorología] Moderate wind warning
  Severity : Moderate
  Tags     : ['Wind']
  Onset    : 2026-04-11T14:00:00
  Expires  : 2026-04-11T21:59:59
  Desc     : Maximum gust of wind: 80 km/h. Viento del noroeste.


/tmp/ipykernel_68017/503701166.py:29: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'onset':       datetime.utcfromtimestamp(a['start']).isoformat() if 'start' in a else '',
/tmp/ipykernel_68017/503701166.py:30: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  'expires':     datetime.utcfromtimestamp(a['end']).isoformat()   if 'end'   in a else '',


In [244]:
# Show alerts as DataFrame
if alerts:
    display(pd.DataFrame(alerts).drop(columns=['description']))
else:
    cols = ['title', 'event', 'sender_name', 'severity', 'urgency', 'certainty', 'onset', 'expires', 'tags']
    print('No active alerts — empty DataFrame schema:')
    print(pd.DataFrame(columns=cols))

,title,event,sender_name,severity,urgency,certainty,onset,expires,tags
0,Moderate wind warning,Moderate wind warning,AEMET. Agencia Estatal de Meteorología,Moderate,Expected,Likely,2026-04-11T14:00:00,2026-04-11T21:59:59,[Wind]


---
## 4. Forecast API — 5 Days / 3-Hour Steps

Separate free-tier endpoint — 40 timestamps over 5 days.  
Used for multi-day risk projection in the risk engine.  
Docs: https://openweathermap.org/forecast5

In [245]:
def get_forecast(api_key, lat, lon, units='metric'):
    """5-day / 3-hour forecast — free tier endpoint."""
    url = 'https://api.openweathermap.org/data/2.5/forecast'
    params = {'lat': lat, 'lon': lon, 'appid': api_key, 'units': units}
    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


def parse_forecast(raw):
    """Convert forecast list to clean records."""
    records = []
    for item in raw['list']:
        weather_list = item.get('weather', [])

        # OWM returns a list but almost always with one element.
        # We take the first (most severe by convention) and log if there are more.
        if len(weather_list) > 1:
            print(f"[NOTE] Multiple weather conditions at {item['dt_txt']}: "
                  f"{[w['id'] for w in weather_list]} — using first.")

        primary_weather = weather_list[0] if weather_list else {}

        records.append({
            'timestamp':           item['dt_txt'],
            'temp_c':              item['main']['temp'],
            'temp_min_c':          item['main']['temp_min'],
            'temp_max_c':          item['main']['temp_max'],
            'humidity_pct':        item['main']['humidity'],
            'wind_speed_ms':       item['wind']['speed'],
            'wind_gust_ms':        item['wind'].get('gust'),
            'rain_3h_mm':          item.get('rain', {}).get('3h', 0),
            'snow_3h_mm':          item.get('snow', {}).get('3h', 0),
            'weather_id':          primary_weather.get('id'),
            'weather_main':        primary_weather.get('main'),
            'weather_description': primary_weather.get('description'),
            'weather_all_ids':     [w['id'] for w in weather_list],  # full list for reference
            'clouds_pct':          item['clouds']['all'],
            'pop':                 item.get('pop', 0),
        })
    return records


raw_forecast     = get_forecast(OWM_API_KEY, LAT, LON)
forecast_records = parse_forecast(raw_forecast)
df_forecast      = pd.DataFrame(forecast_records)
df_forecast['timestamp'] = pd.to_datetime(df_forecast['timestamp'])

print(f'Forecast received: {len(df_forecast)} timestamps')
print(f'Period: {df_forecast["timestamp"].iloc[0]} → {df_forecast["timestamp"].iloc[-1]}')
df_forecast.head(8)

Forecast received: 40 timestamps
Period: 2026-04-10 15:00:00 → 2026-04-15 12:00:00


,timestamp,temp_c,temp_min_c,temp_max_c,humidity_pct,wind_speed_ms,wind_gust_ms,rain_3h_mm,snow_3h_mm,weather_id,weather_main,weather_description,weather_all_ids,clouds_pct,pop
0,2026-04-10 15:00:00,27.34,27.34,27.66,22,1.65,1.14,0.0,0,800,Clear,clear sky,[800],0,0.0
1,2026-04-10 18:00:00,26.38,24.46,26.38,26,3.62,5.09,0.0,0,800,Clear,clear sky,[800],7,0.0
2,2026-04-10 21:00:00,23.36,21.37,23.36,33,2.88,4.00,0.0,0,802,Clouds,scattered clouds,[802],50,0.0
3,2026-04-11 00:00:00,18.95,18.95,18.95,43,2.53,2.06,0.0,0,802,Clouds,scattered clouds,[802],48,0.0
4,2026-04-11 03:00:00,16.20,16.20,16.20,47,2.12,2.28,0.0,0,803,Clouds,broken clouds,[803],72,0.0
5,2026-04-11 06:00:00,16.48,16.48,16.48,47,1.48,1.39,0.0,0,803,Clouds,broken clouds,[803],73,0.0
6,2026-04-11 09:00:00,22.25,22.25,22.25,31,1.04,1.49,0.0,0,804,Clouds,overcast clouds,[804],100,0.0
7,2026-04-11 12:00:00,25.87,25.87,25.87,26,4.37,4.53,0.0,0,804,Clouds,overcast clouds,[804],100,0.0


In [246]:
def weather_id_to_severity(weather_id):
    """
    Convert OWM weather_id to a base severity score (0-40).
    Combined with alert severity in the risk engine.
    Reference: https://openweathermap.org/weather-conditions
    """
    if 200 <= weather_id < 300:
        return 35
    elif 300 <= weather_id < 400:
        return 10
    elif 500 <= weather_id < 600:
        return {500: 15, 501: 20, 502: 30, 503: 35, 504: 40, 511: 25}.get(weather_id, 20)
    elif 600 <= weather_id < 700:
        return 25
    elif weather_id == 781:
        return 40
    elif 700 <= weather_id < 800:
        return 15
    return 0


# Apply severity score per timestamp first
df_forecast['severity_score'] = df_forecast['weather_id'].apply(weather_id_to_severity)

# Ensure date column exists (derived from timestamp)
df_forecast['date'] = df_forecast['timestamp'].dt.date

# Aggregate — worst day = highest severity score
daily_summary = df_forecast.groupby('date').agg(
    temp_min      =('temp_min_c',     'min'),
    temp_max      =('temp_max_c',     'max'),
    wind_max      =('wind_speed_ms',  'max'),
    rain_total    =('rain_3h_mm',     'sum'),
    snow_total    =('snow_3h_mm',     'sum'),
    max_pop       =('pop',            'max'),
    worst_severity=('severity_score', 'max'),
).reset_index()

# Recover the weather_id that produced the worst severity (for labelling)
worst_id_per_day = (
    df_forecast.sort_values('severity_score', ascending=False)
               .groupby('date', as_index=False)['weather_id']
               .first()
               .rename(columns={'weather_id': 'worst_weather_id'})
)

daily_summary = daily_summary.merge(worst_id_per_day, on='date')

print('Daily forecast summary:')
daily_summary

Daily forecast summary:


,date,temp_min,temp_max,wind_max,rain_total,snow_total,max_pop,worst_severity,worst_weather_id
0,2026-04-10,21.37,27.66,3.62,0.00,0,0.0,0,800
1,2026-04-11,9.98,25.87,15.49,0.00,0,0.0,0,804
2,2026-04-12,5.67,7.37,14.65,10.55,0,1.0,20,501
3,2026-04-13,6.15,15.27,14.27,0.00,0,0.0,0,804
4,2026-04-14,9.32,19.63,11.74,0.00,0,0.0,0,800
5,2026-04-15,12.17,21.45,10.54,0.00,0,0.0,0,800


---
## 5. Risk Score Engine — Preliminary

Combines OWM current conditions + One Call alerts into a 0-100 risk score.  
This logic will move to `core/risk_engine.py` in Week 2.

```
risk_score (0-100) = min(weather_severity + alert_severity + wind_bonus + rain_bonus, 100)
```

In [247]:
def severity_to_score(severity_str):
    """Map severity label to numeric score (0-60)."""
    return {'Minor': 10, 'Moderate': 25, 'Severe': 45, 'Extreme': 60}.get(severity_str, 0)


def score_to_level(score):
    """Convert numeric score to risk level label."""
    if score >= 70: return 'CRITICAL'
    if score >= 45: return 'HIGH'
    if score >= 20: return 'ELEVATED'
    return 'LOW'


def compute_risk_score(weather_data, alert_list):
    """
    Combine OWM current data + One Call alerts into a 0-100 risk score.
    Preliminary version — will be refined in Week 2.
    """
    weather_severity = weather_id_to_severity(weather_data['weather_id'])

    alert_severity = max(
        (severity_to_score(a['severity']) for a in alert_list),
        default=0
    )

    wind = weather_data.get('wind_speed_ms', 0) or 0
    wind_bonus = 15 if wind >= 20 else 8 if wind >= 14 else 3 if wind >= 8 else 0

    rain = weather_data.get('rain_1h_mm', 0) or 0
    rain_bonus = 10 if rain >= 20 else 5 if rain >= 10 else 0

    total = min(weather_severity + alert_severity + wind_bonus + rain_bonus, 100)

    return {
        'risk_score': total,
        'risk_level': score_to_level(total),
        'breakdown': {
            'weather_severity': weather_severity,
            'alert_severity':   alert_severity,
            'wind_bonus':       wind_bonus,
            'rain_bonus':       rain_bonus,
        }
    }


risk_result = compute_risk_score(weather, alerts)

print('=' * 42)
print(f"  RISK SCORE : {risk_result['risk_score']}/100")
print(f"  RISK LEVEL : {risk_result['risk_level']}")
print('=' * 42)
print('\nBreakdown:')
for k, v in risk_result['breakdown'].items():
    print(f'  {k:<22} +{v}')

  RISK SCORE : 25/100
  RISK LEVEL : ELEVATED

Breakdown:
  weather_severity       +0
  alert_severity         +25
  wind_bonus             +0
  rain_bonus             +0


In [248]:
# Simulate scenarios to validate thresholds
print('Scenario simulation:\n')

scenarios = [
    {'name': 'Normal day',         'weather_id': 800, 'wind': 3,  'rain': 0,  'alert': 'Unknown'},
    {'name': 'Moderate rain',      'weather_id': 501, 'wind': 6,  'rain': 5,  'alert': 'Minor'},
    {'name': 'Storm + alert',      'weather_id': 502, 'wind': 18, 'rain': 15, 'alert': 'Moderate'},
    {'name': 'Severe emergency',   'weather_id': 212, 'wind': 25, 'rain': 25, 'alert': 'Severe'},
    {'name': 'Catastrophic event', 'weather_id': 781, 'wind': 35, 'rain': 40, 'alert': 'Extreme'},
]

for s in scenarios:
    fw = {'weather_id': s['weather_id'], 'wind_speed_ms': s['wind'], 'rain_1h_mm': s['rain']}
    fa = [{'severity': s['alert']}] if s['alert'] != 'Unknown' else []
    r  = compute_risk_score(fw, fa)
    print(f"  {s['name']:<26} score={r['risk_score']:>3}   level={r['risk_level']}")

Scenario simulation:

  Normal day                 score=  0   level=LOW
  Moderate rain              score= 30   level=ELEVATED
  Storm + alert              score= 68   level=HIGH
  Severe emergency           score=100   level=CRITICAL
  Catastrophic event         score=100   level=CRITICAL


---
## 6. Database Schema

In [249]:
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS weather_readings (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp           TEXT NOT NULL,
    city                TEXT,
    temp_c              REAL,
    feels_like_c        REAL,
    humidity_pct        INTEGER,
    pressure_hpa        INTEGER,
    visibility_m        INTEGER,
    wind_speed_ms       REAL,
    wind_gust_ms        REAL,
    rain_1h_mm          REAL DEFAULT 0,
    snow_1h_mm          REAL DEFAULT 0,
    weather_id          INTEGER,
    weather_main        TEXT,
    weather_desc        TEXT,
    clouds_pct          INTEGER,
    uvi                 REAL,
    created_at          TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS weather_alerts (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    source              TEXT DEFAULT 'aemet',
    title               TEXT,
    event               TEXT,
    sender_name         TEXT,
    severity            TEXT,
    urgency             TEXT,
    certainty           TEXT,
    onset               TEXT,
    expires             TEXT,
    description         TEXT,
    tags                TEXT,
    is_active           INTEGER DEFAULT 1,
    created_at          TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS risk_scores (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp           TEXT NOT NULL,
    score               INTEGER NOT NULL,
    level               TEXT NOT NULL,
    weather_component   INTEGER,
    alert_component     INTEGER,
    wind_bonus          INTEGER,
    rain_bonus          INTEGER,
    created_at          TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS kit_items (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    name                TEXT NOT NULL,
    category            TEXT,
    quantity            REAL NOT NULL,
    unit                TEXT,
    expiry_date         TEXT,
    eu_recommended      REAL,
    notes               TEXT,
    updated_at          TEXT DEFAULT (datetime('now'))
);
"""

conn = sqlite3.connect('prepsense_test.db')
conn.executescript(SCHEMA_SQL)
conn.commit()

tables = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()]
print(f'Database created: prepsense_test.db')
print(f'Tables: {tables}')

Database created: prepsense_test.db
Tables: ['weather_readings', 'sqlite_sequence', 'weather_alerts', 'risk_scores', 'kit_items']


In [250]:
# Insert current weather reading
conn.execute("""
    INSERT INTO weather_readings
    (timestamp, city, temp_c, feels_like_c, humidity_pct, pressure_hpa,
     visibility_m, wind_speed_ms, wind_gust_ms, rain_1h_mm, snow_1h_mm,
     weather_id, weather_main, weather_desc, clouds_pct, uvi)
    VALUES
    (:timestamp, :city, :temp_c, :feels_like_c, :humidity_pct, :pressure_hpa,
     :visibility_m, :wind_speed_ms, :wind_gust_ms, :rain_1h_mm, :snow_1h_mm,
     :weather_id, :weather_main, :weather_description, :clouds_pct, :uvi)
""", weather)

# Insert alerts
for a in alerts:
    conn.execute("""
        INSERT INTO weather_alerts
        (title, event, sender_name, severity, urgency, certainty, onset, expires, description, tags)
        VALUES (:title, :event, :sender_name, :severity, :urgency, :certainty,
                :onset, :expires, :description, :tags)
    """, {**a, 'tags': json.dumps(a['tags'])})

# Insert risk score
conn.execute("""
    INSERT INTO risk_scores
    (timestamp, score, level, weather_component, alert_component, wind_bonus, rain_bonus)
    VALUES (datetime('now'), :risk_score, :risk_level,
            :weather_severity, :alert_severity, :wind_bonus, :rain_bonus)
""", {
    'risk_score':       risk_result['risk_score'],
    'risk_level':       risk_result['risk_level'],
    **risk_result['breakdown']
})

conn.commit()
print('Data inserted into DB')

row = conn.execute('SELECT timestamp, temp_c, wind_speed_ms, weather_desc FROM weather_readings LIMIT 1').fetchone()
print(f'Weather row: {row}')

score_row = conn.execute('SELECT score, level FROM risk_scores LIMIT 1').fetchone()
print(f'Risk score: {score_row[0]}/100 ({score_row[1]})')

Data inserted into DB
Weather row: ('2026-04-10T09:47:53', 9.97, 4.72, 'scattered clouds')
Risk score: 0/100 (LOW)


---
## 7. Emergency Kit — Seed Data

Reference kit based on EU emergency preparedness recommendations (per person, 3 days).  
Source: https://ec.europa.eu/echo/what/civil-protection/eu-emergency-preparedness

In [251]:
EU_KIT_REFERENCE = [
    {'name': 'Drinking water',          'category': 'water',     'eu_recommended': 9.0, 'unit': 'liters'},
    {'name': 'Water purification tabs', 'category': 'water',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Non-perishable food',     'category': 'food',      'eu_recommended': 3.0, 'unit': 'days'},
    {'name': 'Can opener',              'category': 'food',      'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'First aid kit',           'category': 'meds',      'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Regular medication',      'category': 'meds',      'eu_recommended': 7.0, 'unit': 'days'},
    {'name': 'Flashlight',              'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'AA batteries',            'category': 'tools',     'eu_recommended': 6.0, 'unit': 'units'},
    {'name': 'Battery-powered radio',   'category': 'comms',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Power bank',              'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Whistle',                 'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Sleeping bag',            'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Emergency blanket',       'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Document copies',         'category': 'documents', 'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Cash',                    'category': 'documents', 'eu_recommended': 1.0, 'unit': 'units'},
    {'name': 'Basic hygiene kit',       'category': 'tools',     'eu_recommended': 1.0, 'unit': 'units'},
]

# Simulate a realistic kit with intentional gaps and expiry dates
current_kit = [{**item, 'quantity': item['eu_recommended'] * 0.6, 'expiry_date': None}
               for item in EU_KIT_REFERENCE]

current_kit[0]['quantity']    = 4.0           # Water: 4L vs 9L — gap!
current_kit[2]['expiry_date'] = '2026-04-20'  # Food expiring soon
current_kit[5]['expiry_date'] = '2027-01-01'  # Medication OK

for item in current_kit:
    conn.execute("""
        INSERT INTO kit_items (name, category, quantity, unit, expiry_date, eu_recommended)
        VALUES (:name, :category, :quantity, :unit, :expiry_date, :eu_recommended)
    """, item)
conn.commit()

print(f'{len(current_kit)} kit items inserted\n')

df_kit = pd.read_sql(
    'SELECT name, category, quantity, eu_recommended, unit, expiry_date FROM kit_items', conn
)
df_kit['gap']     = df_kit['eu_recommended'] - df_kit['quantity']
df_kit['gap_pct'] = (df_kit['gap'] / df_kit['eu_recommended'] * 100).round(1)
df_kit

16 kit items inserted



,name,category,quantity,eu_recommended,unit,expiry_date,gap,gap_pct
0,Drinking water,water,4.0,9.0,liters,NaN,5.0,55.6
1,Water purification tabs,water,0.6,1.0,units,NaN,0.4,40.0
2,Non-perishable food,food,1.8,3.0,days,2026-04-20,1.2,40.0
3,Can opener,food,0.6,1.0,units,NaN,0.4,40.0
4,First aid kit,meds,0.6,1.0,units,NaN,0.4,40.0
...,...,...,...,...,...,...,...,...
203,Sleeping bag,tools,0.6,1.0,units,NaN,0.4,40.0
204,Emergency blanket,tools,0.6,1.0,units,NaN,0.4,40.0
205,Document copies,documents,0.6,1.0,units,NaN,0.4,40.0
206,Cash,documents,0.6,1.0,units,NaN,0.4,40.0


In [252]:
today = datetime.today()

print('KIT GAPS (below EU recommendation):')
for _, row in df_kit[df_kit['gap'] > 0].iterrows():
    print(f"  {row['name']:<32} {row['quantity']:.1f} / {row['eu_recommended']:.1f} {row['unit']} ({row['gap_pct']}% missing)")

print('\nEXPIRING ITEMS (within 30 days):')
expiring = df_kit[df_kit['expiry_date'].notna()].copy()
expiring['days_to_expiry'] = expiring['expiry_date'].apply(
    lambda d: (datetime.strptime(d, '%Y-%m-%d') - today).days
)
critical = expiring[expiring['days_to_expiry'] <= 30]
for _, row in critical.iterrows():
    flag = 'CRITICAL' if row['days_to_expiry'] <= 7 else 'WARNING'
    print(f"  [{flag}] {row['name']:<32} expires in {row['days_to_expiry']} days ({row['expiry_date']})")
if critical.empty:
    print('  No items expiring within 30 days.')

KIT GAPS (below EU recommendation):
  Drinking water                   4.0 / 9.0 liters (55.6% missing)
  Water purification tabs          0.6 / 1.0 units (40.0% missing)
  Non-perishable food              1.8 / 3.0 days (40.0% missing)
  Can opener                       0.6 / 1.0 units (40.0% missing)
  First aid kit                    0.6 / 1.0 units (40.0% missing)
  Regular medication               4.2 / 7.0 days (40.0% missing)
  Flashlight                       0.6 / 1.0 units (40.0% missing)
  AA batteries                     3.6 / 6.0 units (40.0% missing)
  Battery-powered radio            0.6 / 1.0 units (40.0% missing)
  Power bank                       0.6 / 1.0 units (40.0% missing)
  Whistle                          0.6 / 1.0 units (40.0% missing)
  Sleeping bag                     0.6 / 1.0 units (40.0% missing)
  Emergency blanket                0.6 / 1.0 units (40.0% missing)
  Document copies                  0.6 / 1.0 units (40.0% missing)
  Cash                     

---
## 8. Summary

In [253]:
print('=' * 55)
print('  PREPSENSE — DATA RETRIEVAL VALIDATION SUMMARY')
print('=' * 55)

print('\n[OK] One Call API 3.0 — current conditions')
print(f"     Temp: {weather['temp_c']}C | Wind: {weather['wind_speed_ms']}m/s | {weather['weather_description']}")

print('\n[OK] One Call API 3.0 — national alerts (AEMET)')
print(f'     {len(alerts)} active alert(s)')

print('\n[OK] Forecast API — 5 days / 3h')
print(f'     {len(df_forecast)} timestamps | {len(daily_summary)} days aggregated')

print('\n[OK] Risk Score Engine (preliminary)')
print(f"     Score: {risk_result['risk_score']}/100 ({risk_result['risk_level']})")

print('\n[OK] SQLite DB — schema validated')
print(f'     Tables: weather_readings, weather_alerts, risk_scores, kit_items')

print(f'\n[OK] Emergency kit — {len(current_kit)} items')
gaps_count = len(df_kit[df_kit['gap'] > 0])
print(f'     {gaps_count} gap(s) | {len(critical)} item(s) expiring within 30 days')

print('\n' + '=' * 55)
print('  NEXT STEPS (Week 2)')
print('=' * 55)
print("""
  1. Move fetchers to data/fetchers/ as Python modules
  2. Create data/db.py with DB access functions
  3. Build core/risk_engine.py with logic validated here
  4. Build core/inventory_analyzer.py (gap + expiry detection)
  5. Build core/alert_prioritizer.py (risk x gaps -> ranked actions)
""")

conn.close()
print('DB connection closed.')

  PREPSENSE — DATA RETRIEVAL VALIDATION SUMMARY

[OK] One Call API 3.0 — current conditions
     Temp: 27.34C | Wind: 1.03m/s | clear sky

[OK] One Call API 3.0 — national alerts (AEMET)
     1 active alert(s)

[OK] Forecast API — 5 days / 3h
     40 timestamps | 6 days aggregated

[OK] Risk Score Engine (preliminary)
     Score: 25/100 (ELEVATED)

[OK] SQLite DB — schema validated
     Tables: weather_readings, weather_alerts, risk_scores, kit_items

[OK] Emergency kit — 16 items
     208 gap(s) | 13 item(s) expiring within 30 days

  NEXT STEPS (Week 2)

  1. Move fetchers to data/fetchers/ as Python modules
  2. Create data/db.py with DB access functions
  3. Build core/risk_engine.py with logic validated here
  4. Build core/inventory_analyzer.py (gap + expiry detection)
  5. Build core/alert_prioritizer.py (risk x gaps -> ranked actions)

DB connection closed.


---
## 09. Geopolitical Risk — ACLED Integration

Fetches conflict event data from ACLED for Spain and neighbouring countries,
computes a slow-moving geopolitical score (0-30), and integrates it with the
existing risk score from Section 5.

ACLED registration (free, non-commercial): https://developer.acleddata.com/
Add to your .env:
    ACLED_EMAIL=your@email.com
    ACLED_KEY=your_access_key

This fetcher runs weekly — conflict trends change slowly.

In [256]:
from core.geopolitical_fetcher import (
    get_geopolitical_snapshot,
    build_snapshot,
    ConflictEvent,
    SPAIN_NEIGHBOURS,
)

# Credentials — load from .env or set directly (never commit keys)
from dotenv import load_dotenv
load_dotenv()
ACLED_EMAIL = os.getenv('ACLED_EMAIL', '')
ACLED_PASSWORD = os.getenv('ACLED_PASSWORD', '')

COUNTRY = 'Spain'

print('Geopolitical fetcher ready')
print(f'Neighbours monitored: {SPAIN_NEIGHBOURS}')

Geopolitical fetcher ready
Neighbours monitored: ['France', 'Portugal', 'Morocco', 'Andorra']


In [ ]:
if ACLED_EMAIL and ACLED_PASSWORD:
    try:
        geo_snapshot = get_geopolitical_snapshot(
            country=COUNTRY,
            email=ACLED_EMAIL,
            password=ACLED_PASSWORD,
            include_neighbours=True,
        )
        print(f'Live data fetched: {geo_snapshot.total_events} events')
    except Exception as e:
        print(f'ACLED unavailable ({type(e).__name__}: {e})')
        print('Using simulated snapshot — ACLED account access pending approval.')
        geo_snapshot = build_snapshot(country=COUNTRY, events=[            
                        ConflictEvent('2026-03-01', 'Battles', 'Armed clash',
                          'Spain', 'Cataluña', 'Barcelona', 0, 'Simulated', ''),
                        ConflictEvent('2026-03-15', 'Explosions/Remote violence', 'IED/landmine',
                          'Spain', 'Madrid', 'Madrid', 2, 'Simulated', '')], 
                                      neighbour_events=[
                        ConflictEvent('2026-03-10', 'Battles', 'Armed clash',
                          'France', 'Île-de-France', 'Paris', 1, 'Simulated', '')
                                      ])
else:
    print('No ACLED credentials — using simulated snapshot.')
    geo_snapshot = build_snapshot(country=COUNTRY, events=[], neighbour_events=[])

print(f'\nGeo score : {geo_snapshot.geo_score}/30')
print(f'Trend     : {geo_snapshot.trend}')
print(f'Events    : {geo_snapshot.total_events}')

In [257]:
if ACLED_EMAIL and ACLED_PASSWORD:
    geo_snapshot = get_geopolitical_snapshot(
        country=COUNTRY,
        email=ACLED_EMAIL,
        password=ACLED_PASSWORD,
        include_neighbours=True,
    )
    print(f'Geopolitical snapshot fetched for {geo_snapshot.country}')
else:
    # Simulate a snapshot for testing without credentials
    print('No ACLED credentials found — using simulated snapshot')
    geo_snapshot = build_snapshot(
        country=COUNTRY,
        events=[
            ConflictEvent('2026-03-01', 'Battles', 'Armed clash',
                          'Spain', 'Cataluña', 'Barcelona', 0, 'Simulated', ''),
            ConflictEvent('2026-03-15', 'Explosions/Remote violence', 'IED/landmine',
                          'Spain', 'Madrid', 'Madrid', 2, 'Simulated', ''),
        ],
        neighbour_events=[
            ConflictEvent('2026-03-10', 'Battles', 'Armed clash',
                          'France', 'Île-de-France', 'Paris', 1, 'Simulated', ''),
        ],
    )

print(f'\n{"="*45}')
print(f'  Country        : {geo_snapshot.country}')
print(f'  Period         : {geo_snapshot.period_start} → {geo_snapshot.period_end}')
print(f'  Total events   : {geo_snapshot.total_events}')
print(f'  Total fatalities: {geo_snapshot.total_fatalities}')
print(f'  Event breakdown: {geo_snapshot.event_breakdown}')
print(f'  Trend          : {geo_snapshot.trend}')
print(f'  GEO SCORE      : {geo_snapshot.geo_score}/30')
print(f'{"="*45}')

HTTPError: 403 Client Error: Forbidden for url: https://acleddata.com/api/acled/read?country=Spain&event_date=2026-01-10%7C2026-04-10&event_date_where=BETWEEN&event_type=Battles%7CExplosions%2FRemote+violence%7CViolence+against+civilians&fields=event_date%7Cevent_type%7Csub_event_type%7Ccountry%7Cadmin1%7Clocation%7Cfatalities%7Csource%7Cnotes&_format=json&limit=500

In [258]:
# Add geo_score as a new component to the overall risk score
# Weather risk (0-100) + geopolitical risk (0-30), capped at 100
combined_score = min(risk_result.risk_score + geo_snapshot.geo_score, 100)
combined_level = score_to_level(combined_score)

print('INTEGRATED RISK SCORE\n')
print(f'  Weather + alerts : {risk_result.risk_score:>3}/100  ({risk_result.risk_level})')
print(f'  Geopolitical     : +{geo_snapshot.geo_score:<2}/30   (trend: {geo_snapshot.trend})')
print(f'  {"─"*30}')
print(f'  Combined score   : {combined_score:>3}/100  ({combined_level})')

AttributeError: 'dict' object has no attribute 'risk_score'

In [261]:
import json as json_lib

# Add geo_snapshots table if not exists
conn = sqlite3.connect('prepsense_test.db')
conn.execute("""
    CREATE TABLE IF NOT EXISTS geo_snapshots (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        country          TEXT NOT NULL,
        period_start     TEXT,
        period_end       TEXT,
        total_events     INTEGER,
        total_fatalities INTEGER,
        event_breakdown  TEXT,   -- JSON
        geo_score        INTEGER,
        trend            TEXT,
        fetched_at       TEXT,
        created_at       TEXT DEFAULT (datetime('now'))
    )
""")

conn.execute("""
    INSERT INTO geo_snapshots
    (country, period_start, period_end, total_events, total_fatalities,
     event_breakdown, geo_score, trend, fetched_at)
    VALUES (?,?,?,?,?,?,?,?,?)
""", (
    geo_snapshot.country,
    geo_snapshot.period_start,
    geo_snapshot.period_end,
    geo_snapshot.total_events,
    geo_snapshot.total_fatalities,
    json_lib.dumps(geo_snapshot.event_breakdown),
    geo_snapshot.geo_score,
    geo_snapshot.trend,
    geo_snapshot.fetched_at,
))
conn.commit()

row = conn.execute('SELECT country, geo_score, trend FROM geo_snapshots LIMIT 1').fetchone()
print(f'Stored: {row}')
conn.close()

Stored: ('Spain', 5, 'INCREASING')


In [262]:
pytest.main(['tests/test_geopolitical_fetcher.py', '-v', '--tb=short', '--no-header'])

============================= test session starts ==============================
collecting ... collected 0 items / 1 error

==================================== ERRORS ====================================
_____________ ERROR collecting tests/test_geopolitical_fetcher.py ______________
ImportError while importing test module '/home/ildebrando/code/ijesusjr/000_DS_Portfolio/02_prepsense/tests/test_geopolitical_fetcher.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
../../../../.pyenv/versions/3.12.9/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_geopolitical_fetcher.py:18: in <module>
    from data.fetchers.geopolitical_fetcher import (
E   ModuleNotFoundError: No module named 'data.fetchers'
=========================== short test summary info ============================
ERROR tests/test_geopolitical_fe

<ExitCode.INTERRUPTED: 2>

---
## 12. Health Risk — ECDC CDTR Integration (Option C)

Fetches the latest ECDC Communicable Disease Threats Report (CDTR),
extracts active health threats, and computes an independent health
risk score (0-50).

Source: https://www.ecdc.europa.eu/en/publications-and-data/monitoring/weekly-threats-reports
Published: every Thursday. No authentication required.

Option C design — three independent signals, never summed:
    🌤 Weather Risk    0-100  (fast-moving, OWM One Call)
    ⚔  Geopolitical    0-30   (slow, weekly, ACLED)
    🦠 Health Risk     0-50   (weekly, ECDC CDTR)

In [263]:
from core.health_fetcher import fetch_latest_cdtr_summary, _extract_threats_from_text, _compute_health_score, get_health_snapshot

# Fetch live from ECDC (no credentials needed)
try:
    health_snapshot = get_health_snapshot(timeout=15)
    print(f'Live CDTR fetched: {health_snapshot.week_label}')
except Exception as e:
    print(f'Live fetch failed ({e}) — using simulated snapshot')
    health_snapshot = simulate_health_snapshot('routine')

print(f'\n{"="*50}')
print(f'  HEALTH RISK')
print(f'  Period      : {health_snapshot.period}')
print(f'  Score       : {health_snapshot.health_score}/50')
print(f'  Level       : {health_snapshot.level}')
print(f'  Top threats : {", ".join(health_snapshot.top_threats[:3]) or "None"}')
description = repr(cdtr['description'])
print(f'  Description : {description}')
print(f'  Source      : {health_snapshot.cdtr_url}')
print(f'{"="*50}')

Live CDTR fetched: Week 15, 2026

  HEALTH RISK
  Period      : 4-10 April 2026
  Score       : 19/50
  Level       : ELEVATED
  Top threats : Avian Influenza, Dengue, Influenza
  Description : 'This issue of the ECDC Communicable Disease Threats Report (CDTR) covers the period 4-10 April 2026, and includes updates on avian influenza, dengue, and an overview of respiratory virus epidemiology in the EU/EEA.'
  Source      : https://www.ecdc.europa.eu/en/publications-data/communicable-disease-threats-report-4-10-april-2026-week-15


In [264]:
import sys
for key in list(sys.modules.keys()):
    if 'risk_engine' in key:
        del sys.modules[key]

from core.risk_engine import PrepSenseSignals, _geo_level
print('OK')

OK


In [265]:
from core.risk_engine import RiskResult, PrepSenseSignals, _geo_level

# Rebuild RiskResult from the existing dict
risk_result_obj = RiskResult(
    risk_score=       risk_result['risk_score'],
    risk_level=       risk_result['risk_level'],
    weather_severity= risk_result['breakdown']['weather_severity'],
    alert_severity=   risk_result['breakdown']['alert_severity'],
    wind_bonus=       risk_result['breakdown']['wind_bonus'],
    rain_bonus=       risk_result['breakdown']['rain_bonus'],
)

signals = PrepSenseSignals(
    weather=            risk_result_obj,
    geo_score=          geo_snapshot.geo_score,
    geo_trend=          geo_snapshot.trend,
    geo_country=        geo_snapshot.country,
    health_score=       health_snapshot.health_score,
    health_level=       health_snapshot.level,
    top_health_threats= health_snapshot.top_threats,
)

summary = signals.summary()

print('PREPSENSE — THREE INDEPENDENT RISK SIGNALS\n')
print(f'  🌤  Weather       {summary["weather"]["score"]:>4}/100   {summary["weather"]["level"]}')
print(f'  ⚔   Geopolitical  {summary["geopolitical"]["score"]:>4}/30    {summary["geopolitical"]["level"]}  (trend: {summary["geopolitical"]["trend"]})')
print(f'  🦠  Health        {summary["health"]["score"]:>4}/50    {summary["health"]["level"]}')
if summary["health"]["top_threats"]:
    print(f'       Active threats: {", ".join(summary["health"]["top_threats"][:3])}')
print()
print('  ℹ  Signals are independent — not summed.')

PREPSENSE — THREE INDEPENDENT RISK SIGNALS

  🌤  Weather         25/100   ELEVATED
  ⚔   Geopolitical     5/30    LOW  (trend: INCREASING)
  🦠  Health          19/50    ELEVATED
       Active threats: Avian Influenza, Dengue, Influenza

  ℹ  Signals are independent — not summed.


In [266]:
conn = sqlite3.connect('prepsense_test.db')
conn.execute("""
    CREATE TABLE IF NOT EXISTS health_snapshots (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        week_label      TEXT,
        period          TEXT,
        health_score    INTEGER,
        level           TEXT,
        top_threats     TEXT,   -- JSON array
        cdtr_url        TEXT,
        fetched_at      TEXT,
        created_at      TEXT DEFAULT (datetime('now'))
    )
""")
conn.execute("""
    INSERT INTO health_snapshots
    (week_label, period, health_score, level, top_threats, cdtr_url, fetched_at)
    VALUES (?,?,?,?,?,?,?)
""", (
    health_snapshot.week_label,
    health_snapshot.period,
    health_snapshot.health_score,
    health_snapshot.level,
    json_lib.dumps(health_snapshot.top_threats),
    health_snapshot.cdtr_url,
    health_snapshot.fetched_at,
))
conn.commit()
print('Health snapshot stored in DB')
conn.close()

Health snapshot stored in DB


In [267]:
pytest.main(['tests/test_health_fetcher.py', '-v', '--tb=short', '--no-header'])

============================= test session starts ==============================
collecting ... collected 0 items / 1 error

==================================== ERRORS ====================================
________________ ERROR collecting tests/test_health_fetcher.py _________________
ImportError while importing test module '/home/ildebrando/code/ijesusjr/000_DS_Portfolio/02_prepsense/tests/test_health_fetcher.py'.
Hint: make sure your test modules/packages have valid Python names.
Traceback:
../../../../.pyenv/versions/3.12.9/lib/python3.12/importlib/__init__.py:90: in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tests/test_health_fetcher.py:13: in <module>
    from data.fetchers.health_fetcher import (
E   ModuleNotFoundError: No module named 'data.fetchers'
=========================== short test summary info ============================
ERROR tests/test_health_fetcher.py
!!!!!!!!!!!!!!!

<ExitCode.INTERRUPTED: 2>

---
## 9. Core Modules — Integration Demo

Imports the three core modules and runs them against the real API data
fetched in Sections 2–4. Requires the `core/` folder to be in the same
directory as this notebook.

In [268]:
import sys
import os

# Make sure the notebook can find the core/ package
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))

from core.risk_engine import (
    WeatherSnapshot, Alert, compute_risk_score, score_to_level
)
from core.inventory_analyzer import KitItem, analyze_inventory
from core.alert_prioritizer import prioritize

print('Core modules imported OK')

Core modules imported OK


In [269]:
from datetime import date

# Build WeatherSnapshot from the One Call data fetched in Section 2
weather_snap = WeatherSnapshot(
    weather_id=    weather['weather_id'],
    wind_speed_ms= weather['wind_speed_ms'] or 0.0,
    rain_1h_mm=    weather['rain_1h_mm']    or 0.0,
    wind_gust_ms=  weather['wind_gust_ms']  or 0.0,
)

# Build Alert objects from One Call alerts fetched in Section 3
alert_objects = [
    Alert(event=a['event'], severity=a['severity'], tags=a['tags'])
    for a in alerts
]

# Build KitItem objects from the kit defined in Section 7
kit_objects = []
for item in current_kit:
    exp = item.get('expiry_date')
    kit_objects.append(KitItem(
        name=           item['name'],
        category=       item['category'],
        quantity=       item['quantity'],
        unit=           item['unit'],
        eu_recommended= item['eu_recommended'],
        expiry_date=    date.fromisoformat(exp) if exp else None,
    ))

print(f'WeatherSnapshot : weather_id={weather_snap.weather_id}, wind={weather_snap.wind_speed_ms}m/s')
print(f'Alert objects   : {len(alert_objects)}')
print(f'Kit items       : {len(kit_objects)}')

WeatherSnapshot : weather_id=800, wind=1.03m/s
Alert objects   : 1
Kit items       : 16


In [270]:
# 1. Risk score
risk = compute_risk_score(weather_snap, alert_objects)

# 2. Inventory analysis
inv_report = analyze_inventory(kit_objects)

# 3. Prioritized action list
import sys
for key in list(sys.modules.keys()):
    if 'alert_prioritizer' in key:
        del sys.modules[key]

from core.alert_prioritizer import prioritize

action_list = prioritize(
    risk=             risk_result_obj,
    inventory_report= inv_report,
    geo_score=        geo_snapshot.geo_score,
    geo_trend=        geo_snapshot.trend,
    geo_country=      geo_snapshot.country,
    health_score=     health_snapshot.health_score,
    health_level=     health_snapshot.level,
    top_health_threats= health_snapshot.top_threats,
)

print(f'Total actions: {len(action_list)}\n')
for i, a in enumerate(action_list, 1):
    print(f'{i:>2}. [{a.urgency:<10}] [{a.category:<12}] score={a.priority_score}')
    print(f'    {a.message}')
    if a.detail:
        print(f'    ↳ {a.detail}')
    print()

Total actions: 28

 1. [SOON      ] [KIT_GAP     ] score=55
    Replenish Drinking water: 4.0/9.0 liters (56% missing).
    ↳ Category: water | Priority: HIGH

 2. [SOON      ] [COMBINED    ] score=48
    Weather ELEVATED + low Drinking water stock (4.0/9.0 liters): restock before conditions worsen.
    ↳ Risk score: 25 | Gap: 56% | Amplifier: 1.2x

 3. [SOON      ] [EXPIRY      ] score=45
    Replace Non-perishable food: expires in 10 day(s) (2026-04-20).
    ↳ Urgency: WARNING | Category: food

 4. [SOON      ] [HEALTH_KIT  ] score=42
    Avian Influenza, Dengue active + low First aid kit stock (0.6/1.0 units): restock First aid kit while supplies are available.
    ↳ Health level: ELEVATED | Score: 19/50 | Kit gap: 40%

 5. [SOON      ] [HEALTH_KIT  ] score=42
    Avian Influenza, Dengue active + low Regular medication stock (4.2/7.0 days): restock Regular medication while supplies are available.
    ↳ Health level: ELEVATED | Score: 19/50 | Kit gap: 40%

 6. [SOON      ] [KIT_GAP  

---
## 10. Unit Tests — Inline

Runs the full test suite directly inside the notebook using pytest.
All 62 tests should pass.

In [271]:
import pytest

exit_code = pytest.main([
    'tests/test_core.py',
    '-v',
    '--tb=short',
    '--no-header',
])

print(f'\npytest exit code: {exit_code} (0 = all passed)')

============================= test session starts ==============================
collecting ... collected 62 items

tests/test_core.py::TestWeatherIdToSeverity::test_clear_sky_returns_zero PASSED [  1%]
tests/test_core.py::TestWeatherIdToSeverity::test_cloudy_returns_zero PASSED [  3%]
tests/test_core.py::TestWeatherIdToSeverity::test_light_rain PASSED      [  4%]
tests/test_core.py::TestWeatherIdToSeverity::test_moderate_rain PASSED   [  6%]
tests/test_core.py::TestWeatherIdToSeverity::test_heavy_rain PASSED      [  8%]
tests/test_core.py::TestWeatherIdToSeverity::test_extreme_rain_is_max PASSED [  9%]
tests/test_core.py::TestWeatherIdToSeverity::test_unknown_rain_id_fallback PASSED [ 11%]
tests/test_core.py::TestWeatherIdToSeverity::test_thunderstorm PASSED    [ 12%]
tests/test_core.py::TestWeatherIdToSeverity::test_tornado_is_maximum PASSED [ 14%]
tests/test_core.py::TestWeatherIdToSeverity::test_snow PASSED            [ 16%]
tests/test_core.py::TestWeatherIdToSeverity::test_fog PAS